In [65]:
!unzip src.zip

Archive:  src.zip
  inflating: src/__init__.py         
   creating: src/__pycache__/
  inflating: src/__pycache__/__init__.cpython-313.pyc  
  inflating: src/__pycache__/ews_rules.cpython-313.pyc  
  inflating: src/__pycache__/feature_engineer.cpython-313.pyc  
  inflating: src/__pycache__/forecaster.cpython-313.pyc  
  inflating: src/__pycache__/hierarchical_forecaster.cpython-313.pyc  
  inflating: src/__pycache__/ingestor.cpython-313.pyc  
  inflating: src/__pycache__/ml_trainer.cpython-313.pyc  
  inflating: src/__pycache__/monte_carlo.cpython-313.pyc  
  inflating: src/__pycache__/output_bundler.cpython-313.pyc  
  inflating: src/__pycache__/pipeline.cpython-313.pyc  
  inflating: src/__pycache__/stress_engine.cpython-313.pyc  
  inflating: src/__pycache__/transaction_categorizer.cpython-313.pyc  
  inflating: src/ews_rules.py        
  inflating: src/feature_engineer.py  
  inflating: src/forecaster.py       
  inflating: src/hierarchical_forecaster.py  
  inflating: src/ingesto

# 🏦 Cashflow TFT Training Pipeline


**What this notebook does:**
1. Mounts Google Drive for persistent storage
2. Installs dependencies and clones your repo
4. Builds the PyTorch Forecasting `TimeSeriesDataSet` from your feature store
5. Trains a TFT model with static covariates (city, age, employment)
6. Evaluates with walk-forward CV — MAPE, RMSE, CI coverage
8. Saves the trained model to Google Drive

---


## Cell 1 — Mount Google Drive
Run this first every session. All models, data, and logs persist here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Persistent directory layout on Drive ──────────────────────────────────────
BASE_DIR    = '/content/drive/MyDrive/tft_experiment_temp'
MODEL_DIR   = f'{BASE_DIR}/models'
DATA_DIR    = f'{BASE_DIR}/data'
LOG_DIR     = f'{BASE_DIR}/logs'
COHORT_DIR  = f'{BASE_DIR}/cohort_priors'
ARTIFACT_DIR= f'{BASE_DIR}/artifacts'

for d in [MODEL_DIR, DATA_DIR, LOG_DIR, COHORT_DIR, ARTIFACT_DIR]:
    os.makedirs(d, exist_ok=True)

print('✅ Drive mounted')
print(f'   Base : {BASE_DIR}')
print(f'   Models: {MODEL_DIR}')
print(f'   Data  : {DATA_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted
   Base : /content/drive/MyDrive/tft_experiment_temp
   Models: /content/drive/MyDrive/tft_experiment_temp/models
   Data  : /content/drive/MyDrive/tft_experiment_temp/data


## Cell 2 — Install Dependencies
Runs every session (~4 mins). Output suppressed — check the ✅ at the end.

In [ ]:
%%capture install_output

# Core ML
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install pytorch-forecasting pytorch-lightning mlflow


# Data + utils
!pip install pandas numpy scikit-learn xgboost shap optuna
!pip install prophet  # kept as cold-start fallback

print('✅ All dependencies installed')

In [ ]:
# Verify GPU is available
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Performance settings for Colab T4
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\n→ Training on: {DEVICE}')

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM            : 15.6 GB

→ Training on: cuda


## Cell 3 — Clone Your Repo
Pulls latest code from GitHub every session.

In [ ]:
import subprocess, sys, os

GITHUB_REPO   = 'https://github.com/swaraj2442/z-business.git'  # ← change this to your repo
GITHUB_BRANCH = 'staging/cashflow'  # ← change this to your specific branch
REPO_DIR      = '/content/cashflow_pipeline'

if 'google.colab' in sys.modules:
    if os.path.exists(REPO_DIR):
        print(f'Repo exists. Pulling latest from {GITHUB_BRANCH}...')
        subprocess.run(['git', '-C', REPO_DIR, 'fetch'], capture_output=True)
        subprocess.run(['git', '-C', REPO_DIR, 'checkout', GITHUB_BRANCH], capture_output=True)
        result = subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', GITHUB_BRANCH], capture_output=True, text=True)
        print(f'Pulled latest: {result.stdout.strip()}')
    else:
        print(f'Cloning {GITHUB_BRANCH} branch from {GITHUB_REPO}...')
        result = subprocess.run(['git', 'clone', '-b', GITHUB_BRANCH, GITHUB_REPO, REPO_DIR], capture_output=True, text=True)
        print(f'Cloned: {result.stdout.strip() or result.stderr.strip()}')

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    print(f'✅ Repo ready at {REPO_DIR}')
else:
    if os.path.exists('../src'):
        sys.path.insert(0, os.path.abspath('..'))
    elif os.path.exists('./src'):
        sys.path.insert(0, os.path.abspath('.'))
    print('✅ Running locally. Using local files instead of cloning GitHub.')


Cloning staging/cashflow branch from https://github.com/swaraj2442/z-business.git...
Cloned: Cloning into '/content/cashflow_pipeline'...
fatal: could not read Username for 'https://github.com': No such device or address
✅ Repo ready at /content/cashflow_pipeline


In [66]:
# 1. Generate the massive dataset (takes ~30 seconds)
!python /content/generate_large.py

# 2. Create the /content/data folder just in case it doesn't exist
!mkdir -p /content/data

# 3. Move the fresh CSV files directly into /content/data
!mv /content/transactions_large.csv /content/data/
!mv /content/user_profiles.csv /content/data/

print("✅ Data successfully generated and moved to /content/data!")


Generated 3,062,998 transactions for 2500 users.
  Salaried : 826
  Self-Employed: 870
  Business : 804
Saved to /content/transactions_large.csv and /content/user_profiles.csv
✅ Data successfully generated and moved to /content/data!


In [67]:

# ════════════════════════════════════════════════════════
#  TRAINING CONFIG — tweak as needed
# ════════════════════════════════════════════════════════

CFG = {
    # Data
    'transactions_csv' : '/content/data/transactions_large.csv',
    'min_history_months': 6,

    # TFT architecture
    'max_encoder_length' : 12,   # how many past months TFT looks at
    'max_prediction_length': 6,  # forecast horizon during training
    'hidden_size'        : 32,   # keep small for laptop/Colab Free
    'attention_head_size': 2,
    'dropout'            : 0.1,
    'hidden_continuous_size': 16,

    # Training
    'batch_size'    : 32,
    'max_epochs'    : 50,
    'learning_rate' : 3e-3,
    'gradient_clip' : 0.1,

    # Residual XGBoost
    'xgb_max_depth'   : 3,
    'xgb_n_estimators': 100,
    'min_months_for_residual': 6,  # need at least this many months to fit residual

    # MLflow
    'experiment_name': 'cashflow_tft_v1',
}

print('✅ Config set')
print(f'   Encoder length    : {CFG["max_encoder_length"]} months')
print(f'   Prediction length : {CFG["max_prediction_length"]} months')
print(f'   Hidden size       : {CFG["hidden_size"]}')
print(f'   Batch size        : {CFG["batch_size"]}')
print(f'   Max epochs        : {CFG["max_epochs"]}')

✅ Config set
   Encoder length    : 12 months
   Prediction length : 6 months
   Hidden size       : 32
   Batch size        : 32
   Max epochs        : 50


In [68]:
import mlflow


# Set token for auth (avoids interactive prompt)

# Verify connection
MLFLOW_TRACKING_URI = 'sqlite:///mlflow.db'
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(CFG['experiment_name'])

print(f'   Tracking URI : {MLFLOW_TRACKING_URI}')
print(f'   Experiment   : {CFG["experiment_name"]}')


   Tracking URI : sqlite:///mlflow.db
   Experiment   : cashflow_tft_v1


## Cell 6 — Load Data + Feature Engineering
Runs your existing pipeline modules to build the monthly feature store.

In [69]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

from src.ingestor import load_transactions
from src.transaction_categorizer import categorize_transactions
from src.feature_engineer import build_monthly_features
from tqdm.auto import tqdm

# ── Load raw transactions ─────────────────────────────────────────────────────
raw_df = load_transactions(CFG['transactions_csv'])
print(f'Raw transactions : {len(raw_df):,} rows')
print(f'Entities         : {raw_df["entity_id"].nunique()}')
print(f'Date range       : {raw_df["date"].min().date()} → {raw_df["date"].max().date()}')

# ── Load user profiles ────────────────────────────────────────────────────────
# Assuming your profiles CSV is in the same data folder
profile_path = str(Path(CFG['transactions_csv']).parent / 'user_profiles.csv')
if os.path.exists(profile_path):
    profiles_df = pd.read_csv(profile_path)
    # Map entity_id -> employment_type
    profile_map = profiles_df.set_index('entity_id')['employment_type'].to_dict()
    print(f'Profiles loaded  : {len(profiles_df):,} users')
else:
    profiles_df = None
    profile_map = {}
    print("⚠️ WARNING: user_profiles.csv not found! Features will be degraded.")

# ── Categorize ───────────────────────────────────────────────────────────────
cat_df = categorize_transactions(raw_df)
print(f'\nCategory distribution:')
print(cat_df['category'].value_counts().head(10).to_string())

# ── Build feature store for all entities ─────────────────────────────────────
all_features = []
skipped = []

print("\nGrouping massive dataset (this takes a few seconds)...")
grouped_df = cat_df.groupby('entity_id')

print("Building features per user...")
# Wrap the loop in tqdm for a progress bar!
for entity_id, entity_df in tqdm(grouped_df, total=len(grouped_df)):
    try:
        # 1. Figure out if this user is general (salaried) or msme
        emp_type = profile_map.get(entity_id, 'Self-Employed')

        # 2. Map employment_type to the pipeline's expected entity_type
        if emp_type == 'Salaried':
            etype = 'general'
        else:
            etype = 'msme'

        # 3. Pass BOTH entity_type and profile_df to fix the bug!
        feats = build_monthly_features(
            entity_df,
            entity_id=entity_id,
            entity_type=etype,
            profile_df=profiles_df
        )

        if len(feats) >= CFG['min_history_months']:
            all_features.append(feats)
        else:
            skipped.append((entity_id, len(feats), 'insufficient history'))
    except Exception as e:
        skipped.append((entity_id, 0, str(e)))

feature_store = pd.concat(all_features, ignore_index=True)
feature_store = feature_store.sort_values(['entity_id', 'period']).reset_index(drop=True)

# Add integer time index per entity (required by PyTorch Forecasting)
feature_store['time_idx'] = (
    feature_store.groupby('entity_id')['period']
    .transform(lambda s: (s - s.min()).dt.days // 30)
    .astype(int)
)

print(f'\n✅ Feature store built')
print(f'   Entities in store : {feature_store["entity_id"].nunique()}')
print(f'   Total rows        : {len(feature_store):,}')
print(f'   Columns           : {len(feature_store.columns)}')

# Verification checks
if 'employment_type' in feature_store.columns:
    print("   Profile merge   : ✅ SUCCESS (employment_type found)")
else:
    print("   Profile merge   : ❌ ERROR (employment_type missing)")

salaried_count = feature_store[feature_store['entity_type'] == 'general']['entity_id'].nunique()
msme_count = feature_store[feature_store['entity_type'] == 'msme']['entity_id'].nunique()
print(f'   Salaried users  : {salaried_count:,}')
print(f'   MSME users      : {msme_count:,}')


Raw transactions : 3,062,998 rows
Entities         : 2500
Date range       : 2022-06-30 → 2024-06-30
Profiles loaded  : 2,500 users

Category distribution:
category
utility         893970
other           657912
vendor          652916
food            242251
subscription    241325
transport       241232
emi              60000
sales            40176
salary           19824
gst              13392

Grouping massive dataset (this takes a few seconds)...
Building features per user...


  0%|          | 0/2500 [00:00<?, ?it/s]


✅ Feature store built
   Entities in store : 2500
   Total rows        : 61,995
   Columns           : 72
   Profile merge   : ✅ SUCCESS (employment_type found)
   Salaried users  : 826
   MSME users      : 1,674


In [70]:
import os

# 1. Make sure the folder exists so it doesn't throw an error
os.makedirs('/content/data', exist_ok=True)

# 2. Add the actual file name (.csv) at the end of the path!
csv_path = '/content/data/feature_store.csv'

# 3. Save it
feature_store.to_csv(csv_path, index=False)
print(f'\n📂 Saved CSV for analysis to: {csv_path}')



📂 Saved CSV for analysis to: /content/data/feature_store.csv


## Cell 7 — Add Static + Known Future Features
Enriches the feature store with user profile data and calendar signals that TFT uses as covariates.

In [75]:
import os
import pandas as pd
import numpy as np

# ── Known future features (calendar signals TFT can see into the future) ──────
FESTIVAL_MONTHS = {10, 11}
GST_FILING_MONTHS = {1, 4, 7, 10}
ADVANCE_TAX_MONTHS = {3, 6, 9, 12}

feature_store['is_festival_month']    = feature_store['month'].isin(FESTIVAL_MONTHS).astype(int)
feature_store['is_gst_filing_month']  = feature_store['month'].isin(GST_FILING_MONTHS).astype(int)
feature_store['is_advance_tax_month'] = feature_store['month'].isin(ADVANCE_TAX_MONTHS).astype(int)
feature_store['quarter']              = ((feature_store['month'] - 1) // 3 + 1).astype(str)

# ── Bulletproof Profile Merge ────────────────────────────────────────────────
profile_path = '/content/data/user_profiles.csv'
if os.path.exists(profile_path):
    profiles = pd.read_csv(profile_path)

    # GUARANTEE NO DOUBLE MERGE: If columns already exist, drop them first
    cols_to_drop = [c for c in profiles.columns if c != 'entity_id' and c in feature_store.columns]
    if cols_to_drop:
        feature_store = feature_store.drop(columns=cols_to_drop)

    feature_store = feature_store.merge(profiles, on='entity_id', how='left')
    print(f'✅ User profiles merged safely: {len(profiles)} profiles')
else:
    print('⚠️ WARNING: /content/data/user_profiles.csv NOT FOUND!')

# Fix MLflow illegal characters (+)
if 'age_band' in feature_store.columns:
    feature_store['age_band'] = feature_store['age_band'].astype(str).str.replace('+', '_plus')

# Ensure categorical columns are strings
for col in ['city_tier', 'age_band', 'employment_type', 'quarter', 'entity_id']:
    if col in feature_store.columns:
        feature_store[col] = feature_store[col].astype(str)

# Fill any remaining NaNs in numeric columns
NUMERIC_COLS = [
    'total_inflow', 'total_outflow', 'net_cashflow',
    'emi_to_inflow_ratio', 'fixed_obligation_ratio', 'savings_rate',
    'discretionary_spend_ratio', 'upi_to_inflow_ratio',
    'net_cashflow_lag1', 'net_cashflow_lag2', 'net_cashflow_lag3',
    'net_cashflow_roll3', 'net_cashflow_roll6',
    'inflow_mom_change', 'upi_spend_mom', 'fixed_obligation_mom',
    'cost_of_living_index', 'household_size',
]
for col in NUMERIC_COLS:
    if col in feature_store.columns:
        feature_store[col] = feature_store[col].replace([np.inf, -np.inf], 0).fillna(0).astype(float)

# Re-split
max_time_idx = feature_store.groupby('entity_id')['time_idx'].transform('max')
val_cutoff   = max_time_idx - CFG['max_prediction_length']
train_df = feature_store[feature_store['time_idx'] <= val_cutoff].copy()
val_df   = feature_store.copy()

print(f'\n✅ Feature enrichment and split complete')
print(f'   Train shape: {train_df.shape}')
print(f'   Val shape  : {val_df.shape}')


✅ User profiles merged safely: 2500 profiles

✅ Feature enrichment and split complete
   Train shape: (46995, 86)
   Val shape  : (61995, 86)


In [76]:
import numpy as np

# 1. Guarantee no infinities or NaNs exist in the parent dataframe
feature_store = feature_store.replace([np.inf, -np.inf], 0).fillna(0)

# 2. Recreate train_df and val_df from the CLEANED parent dataframe
max_time_idx = feature_store.groupby('entity_id')['time_idx'].transform('max')
val_cutoff   = max_time_idx - CFG['max_prediction_length']

train_df = feature_store[feature_store['time_idx'] <= val_cutoff].copy()
val_df   = feature_store.copy()

print(f"✅ Cleaned datasets recreated! Train: {len(train_df)}, Val: {len(val_df)}")


✅ Cleaned datasets recreated! Train: 46995, Val: 61995


In [77]:
import numpy as np

# Fix infinities
feature_store = feature_store.replace([np.inf, -np.inf], 0).fillna(0)

# Fix MLflow illegal characters (+)
feature_store['age_band'] = feature_store['age_band'].str.replace('+', '_plus')

# Re-split
max_time_idx = feature_store.groupby('entity_id')['time_idx'].transform('max')
val_cutoff   = max_time_idx - CFG['max_prediction_length']
train_df = feature_store[feature_store['time_idx'] <= val_cutoff].copy()
val_df   = feature_store.copy()


## Cell 8 — Train / Val Split + PyTorch Forecasting Dataset
Builds the `TimeSeriesDataSet` that TFT expects. Chronological split — no leakage.

In [ ]:
!pip install "optuna-integration[pytorch_lightning]"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.4/103.4 kB 8.9 MB/s eta 0:00:00


In [78]:
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

# ── Chronological split ───────────────────────────────────────────────────────
# Use the last max_prediction_length months of each entity as validation
max_time_idx = feature_store.groupby('entity_id')['time_idx'].transform('max')
val_cutoff   = max_time_idx - CFG['max_prediction_length']

train_df = feature_store[feature_store['time_idx'] <= val_cutoff].copy()
val_df   = feature_store.copy()  # full data — dataset trims internally

print(f'Train rows : {len(train_df):,}')
print(f'Val rows   : {len(val_df):,}')

# ── Define feature groups for TFT ────────────────────────────────────────────
# TFT needs to know which features it can see into the future vs only the past

TIME_VARYING_KNOWN_REALS = [
    # Calendar — you always know the future month
    'month',
    'is_festival_month',
    'is_gst_filing_month',
    'is_advance_tax_month',
]

TIME_VARYING_UNKNOWN_REALS = [
    # Observed in the past, unknown in the future
    'net_cashflow',
    'total_inflow',
    'total_outflow',
    'emi_to_inflow_ratio',
    'fixed_obligation_ratio',
    'savings_rate',
    'discretionary_spend_ratio',
    'upi_to_inflow_ratio',
    'net_cashflow_lag1',
    'net_cashflow_lag2',
    'net_cashflow_lag3',
    'net_cashflow_roll3',
    'net_cashflow_roll6',
    'inflow_mom_change',
    'upi_spend_mom',
    'fixed_obligation_mom',
]

STATIC_CATEGORICALS = [
    'employment_type',
    'city_tier',
    'age_band',
]

STATIC_REALS = [
    'household_size',
    'cost_of_living_index',
]

TIME_VARYING_KNOWN_CATEGORICALS = ['quarter']

# Filter to columns that actually exist
def _filter_existing(cols):
    return [c for c in cols if c in feature_store.columns]

TVU_REALS = _filter_existing(TIME_VARYING_UNKNOWN_REALS)
TVK_REALS = _filter_existing(TIME_VARYING_KNOWN_REALS)
S_CATS    = _filter_existing(STATIC_CATEGORICALS)
S_REALS   = _filter_existing(STATIC_REALS)
TVK_CATS  = _filter_existing(TIME_VARYING_KNOWN_CATEGORICALS)

# ── Build TimeSeriesDataSet ───────────────────────────────────────────────────
training_dataset = TimeSeriesDataSet(
    train_df,
    time_idx                  = 'time_idx',
    target                    = 'net_cashflow',
    group_ids                 = ['entity_id'],          # one series per user
    max_encoder_length        = CFG['max_encoder_length'],
    max_prediction_length     = CFG['max_prediction_length'],
    static_categoricals       = S_CATS,
    static_reals              = S_REALS,
    time_varying_known_reals  = TVK_REALS,
    time_varying_known_categoricals = TVK_CATS,
    time_varying_unknown_reals= TVU_REALS,
    target_normalizer         = GroupNormalizer(
        groups=['entity_id'],
    ),
    add_relative_time_idx     = True,   # adds position encoding
    add_target_scales         = True,   # adds mean/std of target as features
    add_encoder_length        = True,   # tells model how much history is available
    allow_missing_timesteps   = True,   # handles cold-start users with gaps
)

# Validation dataset — same params, applied to full data
validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    val_df,
    predict=True,
    stop_randomization=True,
)

# DataLoaders
train_loader = training_dataset.to_dataloader(
    train=True,
    batch_size=CFG['batch_size'],
    num_workers=2,
    persistent_workers=True,
)
val_loader = validation_dataset.to_dataloader(
    train=False,
    batch_size=CFG['batch_size'] * 2,
    num_workers=2,
    persistent_workers=True,
)

print(f'\n✅ Datasets built')
print(f'   Training samples   : {len(training_dataset)}')
print(f'   Validation samples : {len(validation_dataset)}')
print(f'   Static categoricals: {S_CATS}')
print(f'   Static reals       : {S_REALS}')
print(f'   Known future reals : {TVK_REALS}')
print(f'   Unknown past reals : {TVU_REALS}')

Train rows : 46,995
Val rows   : 61,995

✅ Datasets built
   Training samples   : 4495
   Validation samples : 2500
   Static categoricals: ['employment_type', 'city_tier', 'age_band']
   Static reals       : ['household_size', 'cost_of_living_index']
   Known future reals : ['month', 'is_festival_month', 'is_gst_filing_month', 'is_advance_tax_month']
   Unknown past reals : ['net_cashflow', 'total_inflow', 'total_outflow', 'emi_to_inflow_ratio', 'fixed_obligation_ratio', 'savings_rate', 'discretionary_spend_ratio', 'net_cashflow_lag1', 'net_cashflow_lag3', 'net_cashflow_roll3', 'net_cashflow_roll6', 'inflow_mom_change']


In [ ]:
import optuna
from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters

print("🚀 Starting Automatic Hyperparameter Search...")
study = optimize_hyperparameters(
    train_loader,
    val_loader,
    model_path='optuna_tft',
    n_trials=20,          #set this to 5 finish faster
    max_epochs=30,
    gradient_clip_val_range=(0.01, 1.0),
    hidden_size_range=(32, 128),            # Forcing it to search for BIG brains
    hidden_continuous_size_range=(16, 64),
    attention_head_size_range=(2, 4),
    learning_rate_range=(0.001, 0.1),
    dropout_range=(0.1, 0.3),
    trainer_kwargs=dict(limit_train_batches=50, accelerator='auto', devices=1),
    reduce_on_plateau_patience=8,
    use_learning_rate_finder=False,
)

print(f'Best trial parameters: {study.best_trial.params}')

# ── THE AUTOMATIC FIX ───────────────────────────────────────────────────
# Inject the best parameters directly into the global CFG dictionary
CFG.update(study.best_trial.params)

print(f'\n✅ CFG automatically updated with mathematically perfect parameters!')
print(f'   New CFG: {CFG}')
print('   You can now run the Training cell!')


[I 2026-07-19 09:57:14,946] A new study created in memory with name: no-name-6668b15b-5d48-49a1-8eba-4751348c2fda
/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/models/temporal_fusion_transformer/tuning.py:176: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  gradient_clip_val = trial.suggest_loguniform(
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilit

🚀 Starting Automatic Hyperparameter Search...


/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/models/temporal_fusion_transformer/tuning.py:202: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  dropout=trial.suggest_uniform("dropout", *dropout_range),
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/models/temporal_fusion_transformer/tuning.p

## Cell 9 — Define TFT Model
Architecture config. Kept small for Colab Free T4 — scales up by increasing `hidden_size`.

In [ ]:
import time
import torch
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import MLFlowLogger
from pytorch_forecasting import TemporalFusionTransformer, QuantileLoss

print('🚀 Building TFT Model & Trainer...')

tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    hidden_size             = CFG['hidden_size'],
    attention_head_size     = CFG['attention_head_size'],
    dropout                 = CFG['dropout'],
    hidden_continuous_size  = CFG['hidden_continuous_size'],
    loss                    = QuantileLoss(quantiles=[0.1, 0.5, 0.9]),
    learning_rate           = CFG['learning_rate'],
    reduce_on_plateau_patience = 4,
    log_interval            = 10,
    log_val_interval        = 1,
)

checkpoint_path = f'{MODEL_DIR}/tft_best'

checkpoint_callback = ModelCheckpoint(
    dirpath   = checkpoint_path,
    filename  = 'tft-{epoch:02d}-{val_loss:.4f}',
    monitor   = 'val_loss',
    mode      = 'min',
    save_top_k= 1,
    verbose   = True,
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, mode='min', verbose=True),
    LearningRateMonitor(logging_interval='epoch'),
    checkpoint_callback,
]

mlflow_logger = MLFlowLogger(
    experiment_name = CFG['experiment_name'],
    tracking_uri="file:./new_mlruns",
)

trainer = Trainer(
    max_epochs=CFG['max_epochs'],
    accelerator="auto",
    devices=1,
    gradient_clip_val=CFG['gradient_clip_val'],
    callbacks=callbacks,
    logger=mlflow_logger,
    enable_progress_bar=True,
)
print('✅ Setup complete! Ready to train.')


## Cell 10 — Train
This is where the GPU earns its keep. Expected time on T4: ~3–8 mins for 50 epochs at beta scale.

In [ ]:
import time
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pytorch_forecasting import TemporalFusionTransformer

print('─' * 60)
print('🚀 Starting TFT training...')

t0 = time.time()
trainer.fit(
    tft,
    train_dataloaders = train_loader,
    val_dataloaders   = val_loader,
)
elapsed = time.time() - t0

print('─' * 60)
print(f'✅ Training complete in {elapsed/60:.1f} mins')
print(f'   Best epoch     : {trainer.current_epoch}')

score = checkpoint_callback.best_model_score
best_val_loss_str = f"{score.item():.4f}" if score is not None else "N/A"
print(f'   Best val loss  : {best_val_loss_str}')
print(f'   Checkpoint at  : {checkpoint_callback.best_model_path}')

# ── Post-Training Metrics Evaluation ──────────────────────────────────────────
print('\nRunning full evaluation sweep (Train & Val) to calculate metrics...')

best_tft = TemporalFusionTransformer.load_from_checkpoint(checkpoint_callback.best_model_path)
best_tft.eval()

def get_metrics(dataloader, dataset_name):
    print(f"Scoring {dataset_name} Set...")
    results = best_tft.predict(dataloader, mode="quantiles", return_y=True)
    preds  = results.output.cpu().numpy()
    y_true = results.y[0].cpu().numpy().flatten()

    # ── Raw predictions (for true ML accuracy) ────────────────────────────────
    p50_raw = preds[:, :, 1].flatten()
    p10_raw = preds[:, :, 0].flatten()
    p90_raw = preds[:, :, 2].flatten()

    # ── Rounded to nearest ₹100 (for product accuracy — what user sees) ───────
    y_rounded   = np.round(y_true  / 100) * 100
    p50_rounded = np.round(p50_raw / 100) * 100
    p10_rounded = np.round(p10_raw / 100) * 100
    p90_rounded = np.round(p90_raw / 100) * 100

    # ── ML Accuracy metrics (raw vs raw) ──────────────────────────────────────
    wmape_raw = np.sum(np.abs(y_true - p50_raw)) / np.sum(np.abs(y_true)) * 100
    rmse_raw  = np.sqrt(mean_squared_error(y_true, p50_raw))
    mae_raw   = mean_absolute_error(y_true, p50_raw)
    r2_raw    = r2_score(y_true, p50_raw)
    coverage  = np.mean((y_true >= p10_raw) & (y_true <= p90_raw)) * 100

    # ── Product Accuracy metrics (rounded vs rounded) ─────────────────────────
    wmape_prod = np.sum(np.abs(y_rounded - p50_rounded)) / np.sum(np.abs(y_rounded)) * 100
    mae_prod   = mean_absolute_error(y_rounded, p50_rounded)
    r2_prod    = r2_score(y_rounded, p50_rounded)

    print(f'\n── {dataset_name} Metrics (ML Accuracy — Raw vs Raw) ────────────')
    print(f'  WMAPE        : {wmape_raw:.1f}%   (industry standard)')
    print(f'  RMSE         : {rmse_raw:,.0f}')
    print(f'  MAE          : {mae_raw:,.0f}')
    print(f'  R²           : {r2_raw:+.3f}   (1.0 = perfect)')
    print(f'  CI Coverage  : {coverage:.1f}%  (target: >70%)')
    print(f'\n── {dataset_name} Metrics (Product Accuracy — ₹100 Rounded) ─────')
    print(f'  WMAPE        : {wmape_prod:.1f}%')
    print(f'  MAE          : {mae_prod:,.0f}')
    print(f'  R²           : {r2_prod:+.3f}')
    print('────────────────────────────────────────────────────────\n')

    return {
        f"{dataset_name.lower()}_wmape":       wmape_raw,
        f"{dataset_name.lower()}_rmse":        rmse_raw,
        f"{dataset_name.lower()}_mae":         mae_raw,
        f"{dataset_name.lower()}_r2":          r2_raw,
        f"{dataset_name.lower()}_ci_coverage": coverage,
        f"{dataset_name.lower()}_wmape_prod":  wmape_prod,
        f"{dataset_name.lower()}_mae_prod":    mae_prod,
        f"{dataset_name.lower()}_r2_prod":     r2_prod,
    }

train_metrics = get_metrics(train_loader, "Train")
val_metrics   = get_metrics(val_loader,   "Validation")
all_metrics   = {**train_metrics, **val_metrics}

with mlflow.start_run(run_id=mlflow_logger.run_id):
    mlflow.log_metric('training_time_minutes', elapsed / 60)
    mlflow.log_metrics(all_metrics)
    mlflow.log_params(CFG)


## Cell 11 — Load Best Checkpoint + Evaluate
Loads the best checkpoint (lowest val loss) and computes MAPE, RMSE, and CI coverage.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pytorch_forecasting import TemporalFusionTransformer

print('\nRunning full validation sweep to calculate metrics...')

best_tft = TemporalFusionTransformer.load_from_checkpoint(checkpoint_callback.best_model_path)
best_tft.eval()
print(f'✅ Best model loaded')

val_results = best_tft.predict(val_loader, mode="quantiles", return_y=True)
preds  = val_results.output.cpu().numpy()
y_true = val_results.y[0].cpu().numpy().flatten()

# ── Raw predictions (true ML accuracy) ────────────────────────────────────────
p50_raw = preds[:, :, 1].flatten()
p10_raw = preds[:, :, 0].flatten()
p90_raw = preds[:, :, 2].flatten()

# ── Rounded to nearest ₹100 (product accuracy — what the user sees) ───────────
y_rounded   = np.round(y_true  / 100) * 100
p50_rounded = np.round(p50_raw / 100) * 100
p10_rounded = np.round(p10_raw / 100) * 100
p90_rounded = np.round(p90_raw / 100) * 100

# ── ML Accuracy (raw vs raw) ───────────────────────────────────────────────────
wmape_raw = np.sum(np.abs(y_true - p50_raw)) / np.sum(np.abs(y_true)) * 100
rmse_raw  = np.sqrt(mean_squared_error(y_true, p50_raw))
mae_raw   = mean_absolute_error(y_true, p50_raw)
r2_raw    = r2_score(y_true, p50_raw)
coverage  = np.mean((y_true >= p10_raw) & (y_true <= p90_raw)) * 100

# ── Product Accuracy (rounded vs rounded) ─────────────────────────────────────
wmape_prod = np.sum(np.abs(y_rounded - p50_rounded)) / np.sum(np.abs(y_rounded)) * 100
mae_prod   = mean_absolute_error(y_rounded, p50_rounded)
r2_prod    = r2_score(y_rounded, p50_rounded)

print('\n── Validation Metrics (ML Accuracy — Raw vs Raw) ───────────')
print(f'  WMAPE        : {wmape_raw:.1f}%   (industry standard)')
print(f'  RMSE         : {rmse_raw:,.0f}')
print(f'  MAE          : {mae_raw:,.0f}')
print(f'  R²           : {r2_raw:+.3f}   (1.0 = perfect)')
print(f'  CI Coverage  : {coverage:.1f}%  (target: >70%)')
print('\n── Validation Metrics (Product Accuracy — ₹100 Rounded) ────')
print(f'  WMAPE        : {wmape_prod:.1f}%')
print(f'  MAE          : {mae_prod:,.0f}')
print(f'  R²           : {r2_prod:+.3f}')
print('────────────────────────────────────────────────────────────')

metrics = {
    "val_wmape": wmape_raw, "val_rmse": rmse_raw, "val_mae": mae_raw,
    "val_r2": r2_raw, "val_ci_coverage": coverage,
    "val_wmape_prod": wmape_prod, "val_mae_prod": mae_prod, "val_r2_prod": r2_prod,
}

with mlflow.start_run(run_id=mlflow_logger.run_id):
    mlflow.log_metrics(metrics)
    mlflow.log_params(CFG)

# ── Feature Importance ─────────────────────────────────────────────────────────
print('\n── Feature Importance Analysis ──────────────────────────')
interpretation = best_tft.interpret_output(
    best_tft.predict(val_loader, mode='raw', return_x=True)[0],
    reduction='sum',
)

enc_importance_scores = interpretation['encoder_variables'].cpu().numpy()
enc_feature_names     = best_tft.encoder_variables
enc_df = pd.DataFrame({
    'feature': enc_feature_names,
    'importance': enc_importance_scores
}).sort_values('importance', ascending=True)

dec_importance_scores = interpretation['decoder_variables'].cpu().numpy()
dec_feature_names     = best_tft.decoder_variables
dec_df = pd.DataFrame({
    'feature': dec_feature_names,
    'importance': dec_importance_scores
}).sort_values('importance', ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
ax1.barh(enc_df['feature'], enc_df['importance'], color='#4FC3F7')
ax1.set_title("Encoder (Past) Feature Importance", fontsize=14, fontweight='bold')
ax1.set_xlabel("Importance Score")
ax2.barh(dec_df['feature'], dec_df['importance'], color='#EF9A9A')
ax2.set_title("Decoder (Future) Feature Importance", fontsize=14, fontweight='bold')
ax2.set_xlabel("Importance Score")
plt.suptitle('TFT Variable Selection — What the AI Pays Attention To', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n📊 Top 5 Encoder (Past) Features:')
for _, row in enc_df.tail(5).iterrows():
    print(f'   {row["feature"]:25s} → {row["importance"]:.0f}')

print('\n📊 Top 5 Decoder (Future) Features:')
for _, row in dec_df.tail(5).iterrows():
    print(f'   {row["feature"]:25s} → {row["importance"]:.0f}')


## Cell 12 — Attention + Variable Importance
TFT's built-in interpretability — see which features and which past months it's attending to.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ── Variable importance (which features does TFT rely on most?) ───────────────
# Get the raw numerical interpretation scores
interpretation = best_tft.interpret_output(
    best_tft.predict(val_loader, mode='raw', return_x=True)[0],
    reduction='sum',
)

# 1. Encoder Variables (Past Features)
enc_importance_scores = interpretation['encoder_variables'].cpu().numpy()
enc_feature_names = best_tft.encoder_variables

enc_df = pd.DataFrame({
    'feature': enc_feature_names,
    'importance': enc_importance_scores
}).sort_values('importance', ascending=True)

# 2. Decoder Variables (Future Features)
dec_importance_scores = interpretation['decoder_variables'].cpu().numpy()
dec_feature_names = best_tft.decoder_variables

dec_df = pd.DataFrame({
    'feature': dec_feature_names,
    'importance': dec_importance_scores
}).sort_values('importance', ascending=True)

# ── Plotting ────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Plot Encoder Importance
ax1.barh(enc_df['feature'], enc_df['importance'], color='skyblue')
ax1.set_title("Encoder (Past) Feature Importance")
ax1.set_xlabel("Importance Score")

# Plot Decoder Importance
ax2.barh(dec_df['feature'], dec_df['importance'], color='salmon')
ax2.set_title("Decoder (Future) Feature Importance")
ax2.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

# P.S. PyTorch Forecasting also has a built-in auto-plotter if you ever want it!
# best_tft.plot_interpretation(interpretation)


## Cell 13 — XGBoost Personal Residual Layer
Trains a per-user XGBoost on TFT's residuals. Corrects systematic errors for individual users.

In [ ]:
import os
import xgboost as xgb
import joblib
from tqdm.auto import tqdm

RESIDUAL_MODEL_DIR = f'{MODEL_DIR}/residual_models'
os.makedirs(RESIDUAL_MODEL_DIR, exist_ok=True)

RESIDUAL_FEATURES = [
    'net_cashflow_lag1', 'net_cashflow_lag2', 'net_cashflow_lag3',
    'net_cashflow_roll3', 'net_cashflow_roll6',
    'emi_to_inflow_ratio', 'savings_rate', 'fixed_obligation_ratio',
    'inflow_mom_change', 'month',
]

residual_models = {}
residual_report = []

print("Grouping feature store for XGBoost training...")
grouped_features = feature_store.groupby('entity_id')

print(f"Training {len(grouped_features)} personalized XGBoost models...")
# Wrapped in tqdm so you can watch the 2500 models train!
for entity_id, entity_df in tqdm(grouped_features, total=len(grouped_features)):
    entity_df = entity_df.sort_values('period').reset_index(drop=True)
    n = len(entity_df)

    if n < CFG['min_months_for_residual']:
        continue   # not enough history for personal residual

    # Residual = actual - proxy TFT_predicted
    entity_df['tft_pred_proxy'] = entity_df['net_cashflow_roll3'].shift(1).fillna(0)
    entity_df['residual']       = entity_df['net_cashflow'] - entity_df['tft_pred_proxy']

    # Features and target
    avail_features = [f for f in RESIDUAL_FEATURES if f in entity_df.columns]
    X = entity_df[avail_features].fillna(0)
    y = entity_df['residual'].fillna(0)

    # Train/test split (last 2 months held out)
    X_train, X_test = X.iloc[:-2], X.iloc[-2:]
    y_train, y_test = y.iloc[:-2], y.iloc[-2:]

    if len(X_train) < 3:
        continue

    model = xgb.XGBRegressor(
        n_estimators  = CFG['xgb_n_estimators'],
        max_depth     = CFG['xgb_max_depth'],
        learning_rate = 0.05,
        subsample     = 0.8,
        reg_lambda    = 1.0,
        reg_alpha     = 0.1,
        random_state  = 42,
        verbosity     = 0,
    )
    model.fit(X_train, y_train)

    # Score on held-out months
    if len(X_test) > 0:
        residual_rmse = float(np.sqrt(mean_squared_error(y_test, model.predict(X_test))))
        residual_report.append({'entity_id': entity_id, 'residual_rmse': residual_rmse, 'n_months': n})

    # Save model
    model_path = f'{RESIDUAL_MODEL_DIR}/residual_{entity_id}.joblib'
    joblib.dump(model, model_path)
    residual_models[entity_id] = model

print(f'\n✅ Residual models trained: {len(residual_models)} users')
if residual_report:
    report_df = pd.DataFrame(residual_report)
    print(f'   Avg residual RMSE : {report_df["residual_rmse"].mean():,.0f}')
    print(f'   Avg history length: {report_df["n_months"].mean():.1f} months')

    # Log aggregate residual metrics
    with mlflow.start_run(run_id=mlflow_logger.run_id):
        mlflow.log_metric('residual_avg_rmse', report_df['residual_rmse'].mean())
        mlflow.log_metric('residual_n_users', len(residual_models))


Grouping feature store for XGBoost training...
Training 2500 personalized XGBoost models...


  0%|          | 0/2500 [00:00<?, ?it/s]


✅ Residual models trained: 2500 users
   Avg residual RMSE : 94,002
   Avg history length: 24.8 months


## Cell 14 — Hierarchical Blending + Final Inference
Combines TFT global forecast + XGBoost residual correction using history-length-based weights.

In [ ]:
def get_blend_weights(n_months: int) -> dict:
    """Returns blend weights based on how much history the user has."""
    if n_months < 3:
        return {'tft': 0.00, 'residual': 0.00, 'cohort': 1.00}
    elif n_months < 6:
        return {'tft': 0.30, 'residual': 0.10, 'cohort': 0.60}
    elif n_months < 12:
        return {'tft': 0.65, 'residual': 0.35, 'cohort': 0.00}
    elif n_months < 24:
        return {'tft': 0.45, 'residual': 0.55, 'cohort': 0.00}
    else:
        return {'tft': 0.30, 'residual': 0.70, 'cohort': 0.00}


def hierarchical_predict(
    entity_id: str,
    feature_store: pd.DataFrame,
    tft_model,
    residual_models: dict,
    horizon: int = 6,
) -> pd.DataFrame:
    """
    Run hierarchical prediction for one user:
      final = w_tft * tft_forecast + w_residual * residual_correction

    Returns DataFrame with columns:
      period, tft_forecast, residual_correction, final_forecast, p10, p90, weights
    """
    entity_df = feature_store[feature_store['entity_id'] == entity_id].copy()
    n_months  = len(entity_df)
    weights   = get_blend_weights(n_months)

    last_period = entity_df['period'].max()
    future_periods = pd.date_range(
        start=last_period + pd.offsets.MonthBegin(1),
        periods=horizon, freq='MS'
    )

    # ── TFT forecast (returns P10, P50, P90) ─────────────────────────────────
    # In production: run tft_model.predict() on entity's TimeSeriesDataSet
    # Here we use a placeholder — replace with real TFT inference in production
    last_cf     = float(entity_df['net_cashflow'].iloc[-1])
    trend       = float(entity_df['net_cashflow'].diff().mean())
    tft_p50     = np.array([last_cf + trend * (i + 1) for i in range(horizon)])
    residual_std = float(entity_df['net_cashflow'].std()) * 1.28
    tft_p10     = tft_p50 - residual_std
    tft_p90     = tft_p50 + residual_std

    # ── XGBoost residual correction ───────────────────────────────────────────
    residual_correction = np.zeros(horizon)
    if entity_id in residual_models and weights['residual'] > 0:
        avail_features = [f for f in RESIDUAL_FEATURES if f in entity_df.columns]
        last_row = entity_df[avail_features].fillna(0).iloc[[-1]]
        # Apply same last-row features for all horizon steps (simplification)
        # In production: update lag features recursively per step
        for i in range(horizon):
            residual_correction[i] = residual_models[entity_id].predict(last_row)[0]

    # ── Blend ─────────────────────────────────────────────────────────────────
    final_forecast = (
        weights['tft']      * tft_p50 +
        weights['residual'] * (tft_p50 + residual_correction)
    )

    return pd.DataFrame({
        'period'              : future_periods,
        'tft_forecast'        : tft_p50,
        'residual_correction' : residual_correction,
        'final_forecast'      : final_forecast,
        'p10'                 : tft_p10,
        'p90'                 : tft_p90,
        'w_tft'               : weights['tft'],
        'w_residual'          : weights['residual'],
        'n_months_history'    : n_months,
    })


# ── Demo: run for first entity ────────────────────────────────────────────────
sample_entity = feature_store['entity_id'].iloc[0]
n_months = len(feature_store[feature_store['entity_id'] == sample_entity])
weights = get_blend_weights(n_months)

result = hierarchical_predict(
    entity_id      = sample_entity,
    feature_store  = feature_store,
    tft_model      = best_tft,
    residual_models= residual_models,
    horizon        = 6,
)

print(f'✅ Hierarchical forecast for entity: {sample_entity}')
print(f'   History length : {n_months} months')
print(f'   Blend weights  : TFT={weights["tft"]:.0%}  Residual={weights["residual"]:.0%}  Cohort={weights["cohort"]:.0%}')
print()
print(result[['period', 'tft_forecast', 'residual_correction', 'final_forecast', 'p10', 'p90']].to_string(index=False))

NameError: name 'best_model' is not defined

## Cell 15 — Save Everything to Drive + Log Final Artifacts

In [ ]:
import shutil
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M')

# ── Save TFT model ────────────────────────────────────────────────────────────
tft_save_path = f'{MODEL_DIR}/tft_model_{timestamp}.ckpt'
shutil.copy(trainer.checkpoint_callback.best_model_path, tft_save_path)
print(f'✅ TFT model saved: {tft_save_path}')

# ── Save training dataset params (needed to reconstruct dataset for inference)
import pickle
dataset_params_path = f'{MODEL_DIR}/training_dataset_params_{timestamp}.pkl'
with open(dataset_params_path, 'wb') as f:
    pickle.dump(training_dataset.get_parameters(), f)
print(f'✅ Dataset params saved: {dataset_params_path}')

# ── Save feature store ────────────────────────────────────────────────────────
fs_save_path = f'{DATA_DIR}/feature_store_{timestamp}.parquet'
feature_store.to_parquet(fs_save_path, index=False)
print(f'✅ Feature store saved: {fs_save_path}')

# ── Log to MLflow ─────────────────────────────────────────────────────────────
with mlflow.start_run(run_id=mlflow_logger.run_id):
    mlflow.log_artifact(tft_save_path, artifact_path='model')
    mlflow.log_artifact(dataset_params_path, artifact_path='model')
    mlflow.log_param('model_timestamp', timestamp)
    mlflow.log_param('n_entities_trained', feature_store['entity_id'].nunique())
    mlflow.log_param('n_residual_models', len(residual_models))

print()
print('── Summary ─────────────────────────────────────────────────')
print(f'  TFT val MAPE     : {metrics["val_mape"]:.1f}%')
print(f'  TFT val RMSE     : {metrics["val_rmse"]:,.0f}')
print(f'  CI Coverage      : {metrics["val_ci_coverage"]:.1f}%')
print(f'  Residual models  : {len(residual_models)} users')
print(f'  MLflow run ID    : {mlflow_logger.run_id}')
print('────────────────────────────────────────────────────────────')

## Cell 16 — Reload Model (Next Session)
Run this instead of training to load a previously saved model from Drive.

In [ ]:
# ── Uncomment and run this to load a saved model without retraining ────────────

# import pickle, joblib, glob
#
# # Find latest model
# model_files = sorted(glob.glob(f'{MODEL_DIR}/tft_model_*.ckpt'))
# latest_model_path = model_files[-1]
#
# # Find matching dataset params
# ts = latest_model_path.split('tft_model_')[1].replace('.ckpt', '')
# dataset_params_path = f'{MODEL_DIR}/training_dataset_params_{ts}.pkl'
#
# # Rebuild dataset from saved params (needed for inference)
# with open(dataset_params_path, 'rb') as f:
#     dataset_params = pickle.load(f)
#
# # Load TFT
# best_model = TemporalFusionTransformer.load_from_checkpoint(latest_model_path)
# best_model.eval()
#
# # Load residual models
# residual_models = {}
# for path in glob.glob(f'{RESIDUAL_MODEL_DIR}/residual_*.joblib'):
#     entity_id = path.split('residual_')[1].replace('.joblib', '')
#     residual_models[entity_id] = joblib.load(path)
#
# print(f'✅ Loaded TFT from: {latest_model_path}')
# print(f'   Residual models : {len(residual_models)}')

print('Uncomment the block above to load a saved model from Drive.')

In [ ]:
import os
import warnings
import kagglehub
import pandas as pd
import numpy as np
import lightning.pytorch as pl
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer, QuantileLoss

warnings.filterwarnings('ignore')

print("1. Downloading Kaggle Dataset...")
path = kagglehub.dataset_download("khushikyad001/personal-finance-tracker-dataset")

# Find the exact CSV file in the downloaded folder
csv_file = [os.path.join(r, f) for r, d, files in os.walk(path) for f in files if f.endswith('.csv')][0]
df = pd.read_csv(csv_file)
print(f"Dataset loaded with {len(df)} rows.")

print("\n2. Feature Engineering for TFT...")
# Convert date and sort chronologically per user
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['user_id', 'date']).reset_index(drop=True)

# TFT needs a continuous integer time index (0, 1, 2, 3...) per user
df['time_idx'] = df.groupby('user_id').cumcount()

# Ensure categoricals are properly typed as strings
categorical_cols = ['financial_scenario', 'income_type', 'category', 'cash_flow_status', 'financial_stress_level']
for col in categorical_cols:
    df[col] = df[col].astype(str)
df['user_id'] = df['user_id'].astype(str)

# TFT requires entities to have enough history to look back.
# We'll filter for users with at least 4 historical records.
counts = df['user_id'].value_counts()
valid_users = counts[counts >= 4].index
df = df[df['user_id'].isin(valid_users)]
print(f"Filtered to {len(df)} rows ({len(valid_users)} users) with enough history.")

print("\n3. Building TimeSeriesDataSet...")
# Define max history (lookback) and prediction windows
max_encoder_length = 3
max_prediction_length = 1

training_dataset = TimeSeriesDataSet(
    df,
    time_idx="time_idx",
    target="monthly_expense_total",
    group_ids=["user_id"],
    min_encoder_length=1,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,

    # Static features (Things that don't change over time for the user)
    static_categoricals=["user_id", "income_type", "financial_scenario"],

    # Time-varying known features (Things we know in the future, like the time index)
    time_varying_known_reals=["time_idx"],

    # Time-varying unknown features (Things the model has to predict / learn from the past)
    time_varying_unknown_categoricals=["category", "financial_stress_level", "cash_flow_status"],
    time_varying_unknown_reals=[
        "monthly_expense_total",
        "monthly_income",
        "savings_rate",
        "credit_score",
        "debt_to_income_ratio",
        "discretionary_spending",
        "essential_spending"
    ],

    # 👇 The Normalizer that prevents the gradient from exploding! 👇
    target_normalizer=GroupNormalizer(groups=["user_id"]),

    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

train_dataloader = training_dataset.to_dataloader(train=True, batch_size=32, num_workers=0)

print("\n4. Building and Training Temporal Fusion Transformer...")
tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=0.005,        # Slower learning rate for precision
    hidden_size=32,             # Bigger brain
    attention_head_size=2,
    dropout=0.1,
    hidden_continuous_size=16,  # Bigger continuous brain
    loss=QuantileLoss(quantiles=[0.1, 0.5, 0.9]),
    log_interval=5,
)

trainer = pl.Trainer(
    max_epochs=50,              # Full 50 passes over the data
    accelerator="auto",
    devices=1,
    enable_model_summary=True,
)

# Start the training loop!
trainer.fit(
    tft,
    train_dataloaders=train_dataloader
)
print("✅ Done! TFT trained on Kaggle dataset.")


In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('\n── 5. Running Prediction & Evaluation ──────────────────')

# Put the model in evaluation mode so it doesn't try to train anymore
tft.eval()

# Run predictions! (We use train_dataloader since we didn't create a val_dataloader)
results = tft.predict(train_dataloader, mode="quantiles", return_y=True)

# Extract predictions (P10, P50, P90) and actual targets
preds = results.output.cpu().numpy()
y_true = results.y[0].cpu().numpy().flatten()

# We are predicting "monthly_expense_total", so we extract the quantiles
# Rounding to the nearest 10 for clean outputs
p10 = np.round(preds[:, :, 0].flatten() / 10) * 10
p50 = np.round(preds[:, :, 1].flatten() / 10) * 10
p90 = np.round(preds[:, :, 2].flatten() / 10) * 10

# Calculate Metrics
mask = y_true != 0
mape = np.mean(np.abs((y_true[mask] - p50[mask]) / y_true[mask])) * 100 if mask.any() else 0.0
rmse = np.sqrt(mean_squared_error(y_true, p50))
mae = mean_absolute_error(y_true, p50)
r2 = r2_score(y_true, p50)
coverage = np.mean((y_true >= p10) & (y_true <= p90)) * 100

print(f'\n── Kaggle Dataset Metrics ──────────────────────────────')
print(f'  MAPE         : {mape:.1f}%')
print(f'  RMSE         : {rmse:,.0f}')
print(f'  MAE          : {mae:,.0f}')
print(f'  R²           : {r2:+.3f}')
print(f'  CI Coverage  : {coverage:.1f}%')
print('────────────────────────────────────────────────────────')
